# Lab 10: MLP Model for Time-Series Forecasting

**Name:** Zubair Moeen  
**Reg Number:** 22jzele0463  
**Lab:** Machine Learning Lab  
**Lab Supervisor:** Engr.Irshad Ullah  
**University:** UET Peshawar - Campus Nowshera  

## Lab Overview
This notebook develops a Multi-Layer Perceptron model for time-series forecasting. The code loads preprocessed time-series data, defines an MLP architecture, configures checkpoints and monitoring callbacks, trains the model, loads saved results, and evaluates prediction performance using regression metrics.

## Learning Objectives
- Set the working directory and import required machine learning, TensorFlow, and utility modules.
- Define an MLP neural network architecture for time-series input data.
- Configure optimizer, checkpoints, and training monitor callbacks.
- Load train, validation, and test CSV files for forecasting.
- Train and evaluate the model using MAE, MSE, RMSE, R2, and explained variance score.


## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 working path and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utilities.


In [1]:
import os
os.chdir(r'Z:\University\8th Semester\ML Lab\Lab 10,11')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [5]:
model1 = MLP()
model1.summary()

c:\Users\engin\.conda\envs\ml\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        16,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and MLP Architecture
The following cells define time steps, number of features, and the MLP model structure used for forecasting.


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'Z:\University\8th Semester\ML Lab\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
os.path.exists(JSON_PATH)

False

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## Section 3: Checkpoints, Callbacks, and Training Setup
This section configures model checkpoints, training monitor paths, optimizer settings, and compile/training parameters.


In [11]:
import os
path_dataset =r'Z:\University\8th Semester\ML Lab\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\engin\.conda\envs\ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [12]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.007221698760986328 sec


In [13]:
train_X.shape

(835, 24, 21)

In [14]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/60
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2564 - mae: 0.2564 - mape: 130.3202
Epoch 1: val_loss improved from None to 0.07862, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0001-loss0.08.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0001-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.1621 - mae: 0.1621 - mape: 84.4683 - val_loss: 0.0786 - val_mae: 0.0786 - val_mape: 27.0937
Epoch 2/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0832 - mae: 0.0832 - mape: 47.0884 
Epoch 2: val_loss improved from 0.07862 to 0.07549, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0002-loss0.08.h5



Epoch 2: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0002-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0829 - mae: 0.0829 - mape: 46.1648 - val_loss: 0.0755 - val_mae: 0.0755 - val_mape: 23.7693
Epoch 3/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0833 - mae: 0.0833 - mape: 43.1450 
Epoch 3: val_loss did not improve from 0.07549
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0713 - mae: 0.0713 - mape: 38.5569 - val_loss: 0.0952 - val_mae: 0.0952 - val_mape: 27.4761
Epoch 4/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0678 - mae: 0.0678 - mape: 40.5409 
Epoch 4: val_loss did not improve from 0.07549
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0627 - mae: 0.0627 - mape: 35.0141 - val_loss: 0.1300 - val_mae: 0.1300 - val_mape: 46.4407
Epoch 5/60
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0623 - mae: 0.0623 - mape: 32.6630  
Epoch 5: val_loss improved from 0.07549 to 0.04922, saving model to Z:\University\8th Sem


Epoch 5: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0005-loss0.05.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0613 - mae: 0.0613 - mape: 29.0085 - val_loss: 0.0492 - val_mae: 0.0492 - val_mape: 16.8271
Epoch 6/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0553 - mae: 0.0553 - mape: 27.0214 
Epoch 6: val_loss did not improve from 0.04922
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0546 - mae: 0.0546 - mape: 28.5182 - val_loss: 0.0614 - val_mae: 0.0614 - val_mape: 19.6417
Epoch 7/60
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0588 - mae: 0.0588 - mape: 39.1633 
Epoch 7: val_loss did not improve from 0.04922
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0506 - mae: 0.0506 - mape: 32.7549 - val_loss: 0.0932 - val_mae: 0.0932 - val_mape: 31.5044
Epoch 8/60
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0470 - mae: 0.0470 - mape: 25.7451 
Epoch 8: val_loss did not improve from 0.04922
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - l


Epoch 14: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0014-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0397 - mae: 0.0397 - mape: 22.2823 - val_loss: 0.0391 - val_mae: 0.0391 - val_mape: 12.9048
Epoch 15/60
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0484 - mae: 0.0484 - mape: 24.8008 
Epoch 15: val_loss did not improve from 0.03914
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0472 - mae: 0.0472 - mape: 22.1907 - val_loss: 0.0672 - val_mae: 0.0672 - val_mape: 23.7982
Epoch 16/60
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0475 - mae: 0.0475 - mape: 23.6980 
Epoch 16: val_loss did not improve from 0.03914
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0442 - mae: 0.0442 - mape: 22.3521 - val_loss: 0.0561 - val_mae: 0.0561 - val_mape: 17.5964
Epoch 17/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0418 - mae: 0.0418 - mape: 20.2778 
Epoch 17: val_loss did not improve from 0.03914
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/s


Epoch 24: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0024-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0387 - mae: 0.0387 - mape: 18.8854 - val_loss: 0.0363 - val_mae: 0.0363 - val_mape: 12.3230
Epoch 25/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0322 - mae: 0.0322 - mape: 16.4291
Epoch 25: val_loss did not improve from 0.03631
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0369 - mae: 0.0369 - mape: 19.1489 - val_loss: 0.0434 - val_mae: 0.0434 - val_mape: 13.7648
Epoch 26/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0222 - mae: 0.0222 - mape: 11.0636
Epoch 26: val_loss did not improve from 0.03631
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0316 - mae: 0.0316 - mape: 15.7254 - val_loss: 0.0892 - val_mae: 0.0892 - val_mape: 29.8030
Epoch 27/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0701 - mae: 0.0701 - mape: 25.6668
Epoch 27: val_loss did not improve from 0.03631
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


Epoch 50: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0050-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0304 - mae: 0.0304 - mape: 14.4303 - val_loss: 0.0343 - val_mae: 0.0343 - val_mape: 10.2755
Epoch 51/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0281 - mae: 0.0281 - mape: 12.0938
Epoch 51: val_loss did not improve from 0.03432
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0259 - mae: 0.0259 - mape: 10.9009 - val_loss: 0.0419 - val_mae: 0.0419 - val_mape: 13.1621
Epoch 52/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0301 - mae: 0.0301 - mape: 15.8646
Epoch 52: val_loss did not improve from 0.03432
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0334 - mae: 0.0334 - mape: 17.4597 - val_loss: 0.0599 - val_mae: 0.0599 - val_mape: 19.1796
Epoch 53/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0316 - mae: 0.0316 - mape: 11.0865
Epoch 53: val_loss did not improve from 0.03432
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


Epoch 55: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0055-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0268 - mae: 0.0268 - mape: 12.5705 - val_loss: 0.0336 - val_mae: 0.0336 - val_mape: 10.5472
Epoch 56/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0196 - mae: 0.0196 - mape: 8.4163
Epoch 56: val_loss did not improve from 0.03355
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0248 - mae: 0.0248 - mape: 11.0017 - val_loss: 0.0388 - val_mae: 0.0388 - val_mape: 11.0656
Epoch 57/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0194 - mae: 0.0194 - mape: 5.9570
Epoch 57: val_loss did not improve from 0.03355
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0291 - mae: 0.0291 - mape: 14.5841 - val_loss: 0.0386 - val_mae: 0.0386 - val_mape: 11.8988
Epoch 58/60
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0514 - mae: 0.0514 - mape: 14.8358
Epoch 58: val_loss did not improve from 0.03355
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - 

In [15]:
model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0055-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Mean Absolute Error (MAE): 2334.26
Median Absolute Error (MedAE): 2604.53
Mean Squared Error (MSE): 5952890.86
Root Mean Squared Error (RMSE): 2439.85
Mean Absolute Percentage Error (MAPE): 14.99 %
Median Absolute Percentage Error (MDAPE): 16.83 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Section 4: Dataset Loading and Forecast Evaluation
The remaining cells load CSV datasets, prepare forecasting windows, run prediction, and evaluate model performance using regression metrics.


# Fine Tuning

In [16]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0055-loss0.03.h5'
start_epoch= 56

In [17]:
# construct the callback to save only the best model to disk
EpochCheckpoint1 = ModelCheckpoint(
    checkpoints,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

TrainingMonitor1 = TrainingMonitor(
    FIG_PATH,
    jsonPath=JSON_PATH,
    startAt=start_epoch
)

callbacks = [EpochCheckpoint1, TrainingMonitor1]

model_path = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0055-loss0.03.h5'
# model_path = None   # use this if you want to train from scratch

if model_path is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)

    opt = Adam(learning_rate=1e-3)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    print("[INFO] learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0055-loss0.03.h5...
[INFO] learning rate: 9.999999747378752e-05


In [18]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0216 - mae: 0.0216 - mape: 8.8079
Epoch 1: val_loss improved from None to 0.03546, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0001-loss0.04.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0001-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0202 - mae: 0.0202 - mape: 9.6962 - val_loss: 0.0355 - val_mae: 0.0355 - val_mape: 11.1989
Epoch 2/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0180 - mae: 0.0180 - mape: 7.9121 
Epoch 2: val_loss improved from 0.03546 to 0.03529, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0002-loss0.04.h5



Epoch 2: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0002-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0183 - mae: 0.0183 - mape: 7.7963 - val_loss: 0.0353 - val_mae: 0.0353 - val_mape: 10.9754
Epoch 3/10
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0181 - mae: 0.0181 - mape: 7.2417 
Epoch 3: val_loss did not improve from 0.03529
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0180 - mae: 0.0180 - mape: 7.3144 - val_loss: 0.0383 - val_mae: 0.0383 - val_mape: 11.8946
Epoch 4/10
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0180 - mae: 0.0180 - mape: 6.8416 
Epoch 4: val_loss did not improve from 0.03529
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0183 - mae: 0.0183 - mape: 7.7830 - val_loss: 0.0355 - val_mae: 0.0355 - val_mape: 10.7277
Epoch 5/10
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0186 - mae: 0.0186 - mape: 7.3732 
Epoch 5: val_loss improved from 0.03529 to 0.03513, saving model to Z:\University\8th Semester\M


Epoch 5: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0005-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0180 - mae: 0.0180 - mape: 7.3379 - val_loss: 0.0351 - val_mae: 0.0351 - val_mape: 10.8957
Epoch 6/10
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0190 - mae: 0.0190 - mape: 8.4220  
Epoch 6: val_loss did not improve from 0.03513
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0184 - mae: 0.0184 - mape: 7.6894 - val_loss: 0.0375 - val_mae: 0.0375 - val_mape: 11.8543
Epoch 7/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0165 - mae: 0.0165 - mape: 7.1551 
Epoch 7: val_loss did not improve from 0.03513
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0173 - mae: 0.0173 - mape: 7.0783 - val_loss: 0.0363 - val_mae: 0.0363 - val_mape: 10.9706
Epoch 8/10
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0173 - mae: 0.0173 - mape: 6.6693 
Epoch 8: val_loss improved from 0.03513 to 0.03439, saving model to Z:\University\8th Semester\


Epoch 8: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0008-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0173 - mae: 0.0173 - mape: 6.8441 - val_loss: 0.0344 - val_mae: 0.0344 - val_mape: 10.3396
Epoch 9/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0171 - mae: 0.0171 - mape: 7.0145 
Epoch 9: val_loss improved from 0.03439 to 0.03394, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0009-loss0.03.h5



Epoch 9: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0009-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0173 - mae: 0.0173 - mape: 6.7858 - val_loss: 0.0339 - val_mae: 0.0339 - val_mape: 10.4773
Epoch 10/10
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0178 - mae: 0.0178 - mape: 6.7743 
Epoch 10: val_loss did not improve from 0.03394
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0176 - mae: 0.0176 - mape: 7.3920 - val_loss: 0.0364 - val_mae: 0.0364 - val_mape: 10.8948


In [19]:
model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0009-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Mean Absolute Error (MAE): 2752.72
Median Absolute Error (MedAE): 3023.13
Mean Squared Error (MSE): 7989028.17
Root Mean Squared Error (RMSE): 2826.49
Mean Absolute Percentage Error (MAPE): 17.65 %
Median Absolute Percentage Error (MDAPE): 19.53 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, an MLP neural network was implemented for time-series forecasting. The notebook demonstrates model creation, callback configuration, training workflow, and regression-based evaluation.